# Structured Output

Structured output means asking a language model to return data in a predictable format instead of free-form text. A defined structure makes the response easier for a Python program to read, validate, store, and use in later steps.

For example, a movie request can return separate fields such as `title`, `year`, `genre`, `director`, and `rating`. LangChain's `with_structured_output()` method connects the model to a schema that describes these fields.

# Pydantic

Pydantic is a Python library for defining data models with type hints and runtime validation. A Pydantic model describes the fields that should appear in the output and the type expected for each field.

Pydantic models provide useful features such as:
- **Type validation:** checks whether values match their declared types, such as `int`, `str`, or `float`.
- **Required fields:** ensures important fields are present in the response.
- **Field descriptions:** give the model additional guidance about what each field means.
- **Nested models:** allow complex data to contain other structured objects.

When a language model is connected to a Pydantic schema, the result can be returned as a validated Python object instead of an unstructured string.

In [23]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path=r"config\.env")

from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
    model="groq:qwen/qwen3.6-27b",
    temperature=0
)

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x000001ECA936E300>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001ECA93630B0>, model_name='qwen/qwen3.6-27b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [24]:
from pydantic import BaseModel, Field

class Movie(BaseModel): # Pydantic model for structured output
    title: str = Field(..., description="The title of the movie") # ... means the field is required
    year: int = Field(..., description="The release year of the movie")
    genre: str = Field(..., description="The genre of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie (0-10)")

In [25]:
model_structured = model.with_structured_output(Movie)
model_structured

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x000001ECA936E300>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001ECA93630B0>, model_name='qwen/qwen3.6-27b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The release year of the movie', 'type': 'integer'}, 'genre': {'description': 'The genre of the movie', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movie (0-10)', 'type': 'number'}}, 'required': ['title', 'year', 'genre', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': 

In [26]:
model.invoke("Provide the details of the movie 'Inception'")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request**: The user is asking for details about the movie \'Inception\'. This is a straightforward factual request. I need to provide comprehensive but concise information about the film.\n\n2.  **Identify Key Information Needed**:\n   - Title: Inception\n   - Release Year: 2010\n   - Director: Christopher Nolan\n   - Writers: Christopher Nolan\n   - Cast: Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page, Tom Hardy, Ken Watanabe, Dileep Rao, Cillian Murphy, Tom Berenger, Marion Cotillard, Michael Caine\n   - Genre: Sci-Fi, Action, Thriller\n   - Plot Summary: Brief but accurate description of the premise\n   - Themes: Dreams, reality, guilt, subconscious, architecture of dreams\n   - Box Office/Reception: Critical acclaim, awards, commercial success\n   - Notable Elements: Practical effects, score by Hans Zimmer, ambiguous ending, dream-within-a-dream concept\n   - Runtime: ~148 minutes\n   - Product

In [27]:
model_structured.invoke("Provide the details of the movie 'Inception'")

Movie(title='Inception', year=2010, genre='Sci-Fi', director='Christopher Nolan', rating=8.8)

## Message Output with Parsed Structure

By default, a structured-output model returns the parsed result in the schema type, such as a `Movie` Pydantic object. This is the convenient value to use in application code because its fields can be accessed directly, for example with `response.title` or `response.rating`.

Setting `include_raw=True` keeps both views of the response:
- **Raw output:** the original message returned by the language model.
- **Parsed output:** the response converted into the requested schema.
- **Parsing error:** information about a failure to convert the raw response into the schema, when applicable.

Keeping the raw response is useful for debugging and for inspecting how the model's answer was transformed.

In [28]:
class Movie(BaseModel): # Pydantic model for structured output
    """A movie with its details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    genre: str = Field(..., description="The genre of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie (0-10)")

model_structured = model.with_structured_output(Movie, include_raw=True) # include_raw=True will include the raw model output in the response

response = model_structured.invoke("Provide the details of the movie 'Inception'")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:** The user is asking for details about the movie \'Inception\'.\n2.  **Identify Required Information:** I need to provide details like title, year, genre, director, and rating for the movie \'Inception\'.\n3.  **Check Available Tools:** I have a `Movie` tool that takes parameters: `title`, `year`, `genre`, `director`, `rating`. All are required.\n4.  **Determine if Tool is Needed:** The tool seems to be for *creating* or *representing* a movie object, not for *fetching* data from a database. Wait, the description says "A movie with its details." This is a bit ambiguous. Usually, in these prompts, if I have a tool, I should use it to structure the response, or maybe it\'s meant to be called with the known details. Let me recall the actual details of \'Inception\':\n   - Title: Inception\n   - Year: 2010\n   - Genre: Sci-Fi / Action / Thriller\n   - Director: Ch

## Nested Structure

A nested structure is a schema that contains another structured schema. In this example, a `MovieDetails` object contains a list of `Actor` objects in its `cast` field.

Nested models are useful when the data has natural relationships. Each actor has its own `name` and `role`, while the movie has fields such as `title`, `genres`, and `budget`. This keeps related values grouped together and allows Pydantic to validate each actor object inside the list.

The `list[str]` type represents multiple text values, and `float | None` means that `budget` may contain a decimal number or `None` when the budget is unavailable.

In [29]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role played by the actor in the movie")

class MovieDetails(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    cast: list[Actor] = Field(..., description="List of actors in the movie")
    genres: list[str] = Field(..., description="List of genres of the movie")
    budget: float | None = Field(None, description="Budget in millions USD") # Optional field (None means it can be null)

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Miles')], genres=['Science Fiction', 'Action', 'Thriller', 'Adventure'], budget=160.0)

## TypedDict

`TypedDict` is a typing feature that describes the expected keys and value types of a dictionary. It is a simpler alternative to a Pydantic model when you want structured type information without creating a full class.

A `TypedDict` is mainly used for static type checking by tools such as Pylance or mypy. Unlike Pydantic, it does not perform runtime validation when a dictionary is created. This means it is lightweight, but invalid values can still be present unless another part of the application checks them.

Use `TypedDict` for simple dictionary-shaped data when low overhead is important. Use Pydantic when you need runtime validation, field behavior, nested validation, or clear error reporting.

In [30]:
from typing_extensions import Annotated, TypedDict

class MovieDict(TypedDict):
    """A movie with its details."""
    title: Annotated[str, ... ,"The title of the movie"]
    year: Annotated[int, ... ,"The release year of the movie"]
    director: Annotated[str, ... ,"The director of the movie"]
    rating: Annotated[float, ... ,"The rating of the movie (0-10)"]

model_with_typed_dict = model.with_structured_output(MovieDict)
response = model_with_typed_dict.invoke("Provide the details of the movie 'Avengers'")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [31]:
from pydantic import BaseModel, Field

class Actor(TypedDict):
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role played by the actor in the movie")

class MovieDetails(TypedDict):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    cast: list[Actor] = Field(..., description="List of actors in the movie")
    genres: list[str] = Field(..., description="List of genres of the movie")
    budget: float | None = Field(None, description="Budget in millions USD") # Optional field (None means it can be null)

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [ ]:
model.profile # Shows the model's profile, including its capabilities and limitations (e.g., max token limit, supported languages, etc.).

## Dataclasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [39]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Information about a contact."""
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(
    model,
    response_format=ContactInfo # Auto-selects ProviderStrategy based on the model's capabilities
) 

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='ec393f05-317d-4f5a-a51c-2f9c27533a11'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Input text: "John Doe, john@example.com, (555) 123-4567"\n   - Task: Extract contact info\n   - Expected output format: Use the `ContactInfo` function with parameters `name`, `email`, and `phone`.\n\n2.  **Identify Parameters:**\n   - `name`: "John Doe"\n   - `email`: "john@example.com"\n   - `phone`: "(555) 123-4567"\n\n3.  **Validate against Function Schema:**\n   - Function: `ContactInfo`\n   - Required parameters: `name`, `email`, `phone` (all strings)\n   - All parameters are present and match the expected types.\n\n4.  **Construct Function Call:**\n   ```json\n   {\n     "name": "ContactInfo",\n     "arguments": {\n       "name": "John Doe",\n       "email": "

In [40]:
print(result["structured_response"]) # Access the structured output directly

name='John Doe' email='john@example.com' phone='(555) 123-4567'
